In [ ]:
# 1. Mount Google Drive to save the model later safely
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from datasets import load_dataset

# 2. Load the AG News dataset (Using a namespaced version to avoid HF URI bug)
print("Loading dataset...")
dataset = load_dataset("fancyzhx/ag_news")

# 3. Check the dataset (Displaying the 1st news headline for understanding)
print("\n Dataset Loaded Successfully!")
print("Sample Data:")
print(dataset['train'][0])

# Labels meaning: 0 = World, 1 = Sports, 2 = Business, 3 = Sci/Tech

In [ ]:
from transformers import AutoTokenizer

# 1. Load the BERT Tokenizer
print(" Loading BERT Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Define a function to tokenize the text
def tokenize_function(examples):
    # truncation=True ensures no sentence is longer than 128 tokens
    # padding="max_length" makes all sentences the same length
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# 3. Apply the tokenizer to the whole dataset
print(" Tokenizing dataset (this might take a minute)...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 4. Create a smaller subset for fast training!
print(" Creating a smaller subset for faster training...")
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(2000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

print("\n Tokenization and Subsetting Complete!")
print("Train dataset size:", len(small_train_dataset))
print("Test dataset size:", len(small_eval_dataset))

In [ ]:
# 1. Install all required libraries
!pip install -q evaluate scikit-learn transformers datasets accelerate

import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# 2. Load the dataset
print(" Loading dataset...")
dataset = load_dataset("fancyzhx/ag_news")

# 3. Load Tokenizer & Tokenize data
print(" Loading Tokenizer & Tokenizing data...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 4. Create smaller subsets for fast training
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(2000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

# 5. Load Evaluation Metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

# 6. Load Pre-trained BERT Model
print(" Loading BERT Model...")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

# 7. Set up Training Arguments (Saves directly to Drive)
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/DevHub_BERT_News",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# 8. Prepare the Trainer Engine and Train!
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

print(" Starting Training... (Please wait a few minutes)")
trainer.train()

print("\n Training Completed Successfully! Model is saved in your Google Drive.")

In [ ]:
# 1. Pehlay best model aur tokenizer ko exactly main folder mein save kryn!
trainer.save_model("/content/drive/MyDrive/DevHub_BERT_News")
tokenizer.save_pretrained("/content/drive/MyDrive/DevHub_BERT_News")

import gradio as gr
from transformers import pipeline

# 2. Ab dobara model load kryn (ab usay error nahi aayega!)
print(" Loading your fine-tuned model from Drive...")
classifier = pipeline("text-classification", model="/content/drive/MyDrive/DevHub_BERT_News", tokenizer="bert-base-uncased")

# Labels set kar rahay hain
labels_dict = {"LABEL_0": "World", "LABEL_1": "Sports", "LABEL_2": "Business", "LABEL_3": "Sci/Tech"}

def predict_news(text):
    result = classifier(text)[0]
    label = labels_dict[result['label']]
    score = round(result['score'] * 100, 2)
    return f"Category: {label} (Confidence: {score}%)"

# 3. UI bana kar launch kryn!
interface = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=2, placeholder="English news headline yahan likhein..."),
    outputs=gr.Textbox(label="AI Prediction result"),
    title=" AI News Classifier",
    description="Enter any news headline and my AI model will guess its category!"
)

interface.launch(share=True)